# 03 — Normalised Data Model & Validation

**Project:** AI-Powered Customer Churn & Retention Analytics  
**Phase:** 1 — Data Model  

### Context
The join-grain investigation (`JOIN_GRAIN_INVESTIGATION.md`) established that a row-level join  
between `purchases.csv` and `invoice_items.csv` on `(InvoiceID, product_id)` is a **many-to-many join**  
that inflates row counts from 431,173 to 442,587 (+2.65%) and revenue from £9,307,314 to £9,388,019 (+0.87%).

This notebook builds the correct normalised analytical layer:

| Table | Grain | Source |
|-------|-------|--------|
| `dim_invoices` | 1 row per InvoiceID | purchases + invoice_items (aggregated) |
| `fact_invoice_items` | 1 row per line-item | invoice_items + products |
| `dim_customers` | 1 row per CustomerID | customers |
| `dim_products` | 1 row per product_id | products |

> **No EDA, RFM, churn labels, or ML models in this notebook.**  
> **Raw files in `data/raw/` are never written to.**

In [1]:
import sys, os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def _find_project_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / 'src').is_dir():
            return p
    return start

PROJECT_ROOT = _find_project_root(Path(os.getcwd()).resolve())
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s  %(message)s')

import pandas as pd
import json

from data_loader import load_all
from data_model  import (
    build_analytical_layer,
    run_integrity_checks,
)

print('Environment OK. Project root:', PROJECT_ROOT)

Environment OK. Project root: C:\Users\Siddharth Reddy\projects\new\customer-churn-analytics


## Section 1 — Load Cleaned Source Tables

In [2]:
raw_frames = load_all()
print('Raw row counts:')
for name, df in raw_frames.items():
    print(f'  {name:20s}: {len(df):>8,} rows')

INFO  Loading customers from C:\Users\Siddharth Reddy\projects\new\customer-churn-analytics\data\raw\customers.csv


INFO    → 8237 rows × 2 columns


INFO  Loading purchases from C:\Users\Siddharth Reddy\projects\new\customer-churn-analytics\data\raw\purchases.csv


INFO    → 436689 rows × 5 columns


INFO    purchases.date parsed OK  (range: 2014-01-01 – 2015-12-30)


INFO  Loading invoice_items from C:\Users\Siddharth Reddy\projects\new\customer-churn-analytics\data\raw\invoice_items.csv


INFO    → 436689 rows × 5 columns


INFO  Loading products from C:\Users\Siddharth Reddy\projects\new\customer-churn-analytics\data\raw\products.csv


INFO    → 4033 rows × 4 columns


Raw row counts:
  customers           :    8,237 rows
  purchases           :  436,689 rows
  invoice_items       :  436,689 rows
  products            :    4,033 rows


## Section 2 — The Data-Grain Problem

### Why purchases ↔ invoice_items must NOT be joined at row level

**`purchases.csv`** stores: `InvoiceID, CustomerID, date, product_id, quantity`  
Its grain (smallest unique key) is `(InvoiceID, product_id, quantity)` — the same product can appear  
multiple times within one invoice at different quantities.

**`invoice_items.csv`** stores: `InvoiceID, product_id, quantity, price, line_total`  
Its grain is `(InvoiceID, product_id, quantity, price, line_total)` — the same product can appear  
multiple times at the same quantity but at different prices.

Because `purchases` has no price column, it cannot distinguish multi-price rows. When these two files  
are independently deduplicated, `invoice_items` retains more rows for those multi-price keys,  
creating a many-to-many join on `(InvoiceID, product_id)` that inflates every downstream aggregation.

**Safe pattern:**  
```
invoice_items  →  aggregate to InvoiceID  →  join InvoiceID context from purchases
```
Never the reverse.

In [3]:
# Demonstrate the grain issue concisely
from data_loader import load_purchases, load_invoice_items

purch_raw = load_purchases()
ii_raw    = load_invoice_items()

purch = purch_raw.drop_duplicates()
ii    = ii_raw.drop_duplicates()

for label, df, keys in [
    ('purchases',     purch, ['InvoiceID','product_id']),
    ('purchases',     purch, ['InvoiceID','product_id','quantity']),
    ('invoice_items', ii,    ['InvoiceID','product_id']),
    ('invoice_items', ii,    ['InvoiceID','product_id','quantity']),
    ('invoice_items', ii,    ['InvoiceID','product_id','quantity','price','line_total']),
]:
    n_keys = df.groupby(keys).size()
    unique = (n_keys == 1).all()
    print(f'  {label:16s}  key={str(keys):55s}  unique={str(unique):5s}  max_per_key={n_keys.max()}')

INFO  Loading purchases from C:\Users\Siddharth Reddy\projects\new\customer-churn-analytics\data\raw\purchases.csv


INFO    → 436689 rows × 5 columns


INFO    purchases.date parsed OK  (range: 2014-01-01 – 2015-12-30)


INFO  Loading invoice_items from C:\Users\Siddharth Reddy\projects\new\customer-churn-analytics\data\raw\invoice_items.csv


INFO    → 436689 rows × 5 columns


  purchases         key=['InvoiceID', 'product_id']                              unique=False  max_per_key=4
  purchases         key=['InvoiceID', 'product_id', 'quantity']                  unique=True   max_per_key=1
  invoice_items     key=['InvoiceID', 'product_id']                              unique=False  max_per_key=4


  invoice_items     key=['InvoiceID', 'product_id', 'quantity']                  unique=False  max_per_key=4
  invoice_items     key=['InvoiceID', 'product_id', 'quantity', 'price', 'line_total']  unique=True   max_per_key=1


## Section 3 — Build dim_invoices

In [4]:
# Verify pre-conditions manually (no join needed for this)
inv_check = purch.groupby('InvoiceID').agg(
    n_customers=('CustomerID','nunique'),
    n_dates=('date','nunique')
)
multi_cust  = (inv_check['n_customers'] > 1).sum()
multi_dates = (inv_check['n_dates']     > 1).sum()
print(f'Invoices with >1 CustomerID: {multi_cust}  {"✅" if multi_cust == 0 else "❌"}')
print(f'Invoices with >1 date:       {multi_dates}  {"✅" if multi_dates == 0 else "❌"}')

Invoices with >1 CustomerID: 0  ✅
Invoices with >1 date:       0  ✅


In [5]:
# Build invoice context (customer + date) from purchases
invoice_map = (
    purch.groupby('InvoiceID')
         .agg(CustomerID=('CustomerID','first'),
              date=('date','first'))
         .reset_index()
)
print(f'invoice_map: {invoice_map.shape}  (should be 33,499 rows)')

# Aggregate invoice_items to invoice level — NO join to purchases
invoice_rev = (
    ii.groupby('InvoiceID')
      .agg(invoice_revenue=('line_total','sum'),
           line_item_count=('line_total','count'))
      .reset_index()
)
print(f'invoice_rev:  {invoice_rev.shape}  (should be 33,499 rows)')

# Merge on InvoiceID ONLY
dim_inv = invoice_map.merge(invoice_rev, on='InvoiceID', how='left')
print(f'\ndim_invoices: {dim_inv.shape}')
print(f'  Unique InvoiceIDs: {dim_inv["InvoiceID"].nunique()}')
print(f'  Unique CustomerIDs: {dim_inv["CustomerID"].nunique()}')
print(f'  Total invoice revenue: £{dim_inv["invoice_revenue"].sum():,.2f}')
print(f'\nFirst 5 rows:')
dim_inv.head()

invoice_map: (33499, 3)  (should be 33,499 rows)


invoice_rev:  (33499, 3)  (should be 33,499 rows)

dim_invoices: (33499, 5)
  Unique InvoiceIDs: 33499
  Unique CustomerIDs: 8237
  Total invoice revenue: £9,307,314.40

First 5 rows:


,InvoiceID,CustomerID,date,invoice_revenue,line_item_count
0,1,1000,2015-03-15,29.19,4
1,2,1000,2014-06-24,18.30,3
2,3,1000,2015-07-24,26.10,2
3,4,1000,2015-11-25,42.32,2
4,5,1000,2015-05-27,30.96,2


## Section 4 — Build fact_invoice_items

In [6]:
from data_loader import load_products
prod = load_products().drop_duplicates()

# Attach product descriptors
fact_ii = ii.merge(
    prod[['product_id','item','category']],
    on='product_id',
    how='left'
)
print(f'fact_invoice_items: {fact_ii.shape}')
print(f'  Unique InvoiceIDs: {fact_ii["InvoiceID"].nunique()}')
print(f'  Unique product_ids: {fact_ii["product_id"].nunique()}')
print(f'  Zero-price rows: {(fact_ii["price"]==0).sum()}')
print(f'  Total line_total: £{fact_ii["line_total"].sum():,.2f}')
print(f'\nFirst 5 rows:')
fact_ii.head()

INFO  Loading products from C:\Users\Siddharth Reddy\projects\new\customer-churn-analytics\data\raw\products.csv


INFO    → 4033 rows × 4 columns


fact_invoice_items: (431263, 7)
  Unique InvoiceIDs: 33499
  Unique product_ids: 4033
  Zero-price rows: 40
  Total line_total: £9,307,314.40

First 5 rows:


,InvoiceID,product_id,quantity,price,line_total,item,category
0,536365,3851,6,2.55,15.30,white hanging heart t-light holder,Home Decor
1,536365,3859,6,3.39,20.34,white metal lantern,Home Decor
2,536365,885,8,2.75,22.00,cream cupid hearts coat hanger,"Health, Beauty & Personal Care"
3,536365,1873,6,3.39,20.34,knitted union flag hot water bottle,Party & Festive
4,536365,2871,6,3.39,20.34,red woolly hottie white heart.,"Health, Beauty & Personal Care"


## Section 5 — Build dim_customers

In [7]:
from data_loader import load_customers
dim_cust = load_customers().drop_duplicates(subset='CustomerID').reset_index(drop=True)
print(f'dim_customers: {dim_cust.shape}')
print(dim_cust['customer_type'].value_counts().to_string())
dim_cust.head()

INFO  Loading customers from C:\Users\Siddharth Reddy\projects\new\customer-churn-analytics\data\raw\customers.csv


INFO    → 8237 rows × 2 columns


dim_customers: (8237, 2)
customer_type
private       7161
wholesaler    1076


,CustomerID,customer_type
0,17850,wholesaler
1,13047,wholesaler
2,12583,wholesaler
3,13748,private
4,15100,private


## Section 6 — Build dim_products

In [8]:
dim_prod = prod[['product_id','item','category','price']].drop_duplicates(subset='product_id').reset_index(drop=True)
print(f'dim_products: {dim_prod.shape}')
print(f'  Categories: {dim_prod["category"].nunique()}')
dim_prod.head()

dim_products: (4033, 4)
  Categories: 25


,product_id,item,category,price
0,3851,white hanging heart t-light holder,Home Decor,2.55
1,3859,white metal lantern,Home Decor,3.39
2,885,cream cupid hearts coat hanger,"Health, Beauty & Personal Care",2.75
3,1873,knitted union flag hot water bottle,Party & Festive,3.39
4,2871,red woolly hottie white heart.,"Health, Beauty & Personal Care",3.39


## Section 7 — Referential Integrity Checks

In [9]:
checks = []

def chk(name, passed, detail):
    icon = '✅' if passed else '❌'
    checks.append({'check': name, 'passed': passed, 'detail': detail})
    print(f'{icon} [{"PASS" if passed else "FAIL"}] {name}')
    print(f'       {detail}')

# A — dim_invoices
chk('A1 dim_invoices InvoiceID unique',
    dim_inv['InvoiceID'].nunique() == len(dim_inv),
    f'unique={dim_inv["InvoiceID"].nunique()}  rows={len(dim_inv)}')

chk('A2 dim_invoices CustomerID not null',
    dim_inv['CustomerID'].isna().sum() == 0,
    f'nulls={dim_inv["CustomerID"].isna().sum()}')

chk('A3 dim_invoices date not null',
    dim_inv['date'].isna().sum() == 0,
    f'nulls={dim_inv["date"].isna().sum()}')

chk('A4 dim_invoices 1 customer per invoice',
    dim_inv.groupby('InvoiceID')['CustomerID'].nunique().max() == 1,
    'max customers per invoice = ' + str(dim_inv.groupby('InvoiceID')['CustomerID'].nunique().max()))

chk('A5 dim_invoices 1 date per invoice',
    dim_inv.groupby('InvoiceID')['date'].nunique().max() == 1,
    'max dates per invoice = ' + str(dim_inv.groupby('InvoiceID')['date'].nunique().max()))

# B — fact_invoice_items
chk('B1 fact_ii no exact duplicates',
    fact_ii[['InvoiceID','product_id','quantity','price','line_total']].duplicated().sum() == 0,
    f'dups={fact_ii[["InvoiceID","product_id","quantity","price","line_total"]].duplicated().sum()}')

chk('B2 fact_ii InvoiceID not null',
    fact_ii['InvoiceID'].isna().sum() == 0,
    f'nulls={fact_ii["InvoiceID"].isna().sum()}')

chk('B3 fact_ii product_id not null',
    fact_ii['product_id'].isna().sum() == 0,
    f'nulls={fact_ii["product_id"].isna().sum()}')

chk('B4 fact_ii quantity >= 0',
    (fact_ii['quantity'] < 0).sum() == 0,
    f'negative_qty={(fact_ii["quantity"] < 0).sum()}')

chk('B5 fact_ii price >= 0',
    (fact_ii['price'] < 0).sum() == 0,
    f'negative_price={(fact_ii["price"] < 0).sum()}')

chk('B6 fact_ii line_total >= 0',
    (fact_ii['line_total'] < 0).sum() == 0,
    f'negative_line_total={(fact_ii["line_total"] < 0).sum()}')

# C — referential integrity
inv_dim  = set(dim_inv['InvoiceID'])
inv_fact = set(fact_ii['InvoiceID'])
chk('C1 all dim_invoices IDs exist in fact_ii',
    len(inv_dim - inv_fact) == 0,
    f'orphans_in_dim={len(inv_dim - inv_fact)}')

chk('C2 all fact_ii IDs exist in dim_invoices',
    len(inv_fact - inv_dim) == 0,
    f'orphans_in_fact={len(inv_fact - inv_dim)}')

chk('C3 dim_invoices.CustomerID in dim_customers',
    len(set(dim_inv['CustomerID']) - set(dim_cust['CustomerID'])) == 0,
    f'orphans={len(set(dim_inv["CustomerID"]) - set(dim_cust["CustomerID"]))}')

chk('C4 fact_ii.product_id in dim_products',
    len(set(fact_ii['product_id']) - set(dim_prod['product_id'])) == 0,
    f'orphans={len(set(fact_ii["product_id"]) - set(dim_prod["product_id"]))}')

✅ [PASS] A1 dim_invoices InvoiceID unique
       unique=33499  rows=33499
✅ [PASS] A2 dim_invoices CustomerID not null
       nulls=0
✅ [PASS] A3 dim_invoices date not null
       nulls=0
✅ [PASS] A4 dim_invoices 1 customer per invoice
       max customers per invoice = 1
✅ [PASS] A5 dim_invoices 1 date per invoice
       max dates per invoice = 1


✅ [PASS] B1 fact_ii no exact duplicates


       dups=0
✅ [PASS] B2 fact_ii InvoiceID not null
       nulls=0
✅ [PASS] B3 fact_ii product_id not null
       nulls=0
✅ [PASS] B4 fact_ii quantity >= 0
       negative_qty=0
✅ [PASS] B5 fact_ii price >= 0
       negative_price=0
✅ [PASS] B6 fact_ii line_total >= 0
       negative_line_total=0
✅ [PASS] C1 all dim_invoices IDs exist in fact_ii
       orphans_in_dim=0
✅ [PASS] C2 all fact_ii IDs exist in dim_invoices
       orphans_in_fact=0
✅ [PASS] C3 dim_invoices.CustomerID in dim_customers
       orphans=0


✅ [PASS] C4 fact_ii.product_id in dim_products
       orphans=0


## Section 8 — Revenue Reconciliation

In [10]:
# Authoritative revenue: direct from deduplicated invoice_items, no join
rev_direct   = ii['line_total'].sum()
# Revenue via dim_invoices aggregation
rev_dim_inv  = dim_inv['invoice_revenue'].sum()
# Revenue from fact_invoice_items (should match direct)
rev_fact_ii  = fact_ii['line_total'].sum()

print('=== REVENUE RECONCILIATION ===')
print(f'  Direct from invoice_items (no join):  £{rev_direct:>14,.2f}  ← AUTHORITATIVE')
print(f'  From dim_invoices (aggregated):        £{rev_dim_inv:>14,.2f}')
print(f'  From fact_invoice_items:               £{rev_fact_ii:>14,.2f}')
print(f'  dim_inv vs direct diff:                £{rev_dim_inv - rev_direct:>14,.4f}')
print(f'  fact_ii vs direct diff:                £{rev_fact_ii - rev_direct:>14,.4f}')
print()

match_dim = abs(rev_dim_inv - rev_direct) < 0.01
match_fact = abs(rev_fact_ii - rev_direct) < 0.01
print(f'  dim_invoices revenue matches direct:   {"✅ PASS" if match_dim  else "❌ FAIL"}')
print(f'  fact_invoice_items revenue matches:    {"✅ PASS" if match_fact else "❌ FAIL"}')

chk('D1 revenue dim_invoices == direct from ii',  match_dim,
    f'dim=£{rev_dim_inv:,.2f}  direct=£{rev_direct:,.2f}  diff=£{rev_dim_inv-rev_direct:,.4f}')
chk('D2 revenue fact_ii == direct from ii',        match_fact,
    f'fact=£{rev_fact_ii:,.2f}  direct=£{rev_direct:,.2f}  diff=£{rev_fact_ii-rev_direct:,.4f}')

=== REVENUE RECONCILIATION ===
  Direct from invoice_items (no join):  £  9,307,314.40  ← AUTHORITATIVE
  From dim_invoices (aggregated):        £  9,307,314.40
  From fact_invoice_items:               £  9,307,314.40
  dim_inv vs direct diff:                £        0.0000
  fact_ii vs direct diff:                £        0.0000

  dim_invoices revenue matches direct:   ✅ PASS
  fact_invoice_items revenue matches:    ✅ PASS
✅ [PASS] D1 revenue dim_invoices == direct from ii
       dim=£9,307,314.40  direct=£9,307,314.40  diff=£0.0000
✅ [PASS] D2 revenue fact_ii == direct from ii
       fact=£9,307,314.40  direct=£9,307,314.40  diff=£0.0000


## Section 9 — Invoice and Customer Count Validation

In [11]:
n_inv_fact  = fact_ii['InvoiceID'].nunique()
n_inv_dim   = len(dim_inv)
n_cust_dim  = dim_inv['CustomerID'].nunique()
n_cust_cust = len(dim_cust)

print('=== INVOICE / CUSTOMER COUNTS ===')
print(f'  fact_ii unique InvoiceIDs : {n_inv_fact:>8,}')
print(f'  dim_invoices rows          : {n_inv_dim:>8,}')
count_match = n_inv_fact == n_inv_dim
print(f'  Match                      : {"✅ PASS" if count_match else "❌ FAIL — explain below"}')
if not count_match:
    only_dim  = len(inv_dim  - inv_fact)
    only_fact = len(inv_fact - inv_dim)
    print(f'    InvoiceIDs only in dim_invoices : {only_dim}')
    print(f'    InvoiceIDs only in fact_ii      : {only_fact}')

print()
print(f'  Unique customers in dim_invoices : {n_cust_dim:>8,}')
print(f'  Rows in dim_customers            : {n_cust_cust:>8,}')
print(f'  All invoice customers in dim_cust: {"✅ PASS" if n_cust_dim <= n_cust_cust else "❌ FAIL"}')

chk('E1 invoice count fact_ii == dim_invoices', count_match,
    f'fact_ii unique={n_inv_fact}  dim_invoices rows={n_inv_dim}')
chk('F1 all invoice customers in dim_customers', n_cust_dim <= n_cust_cust,
    f'inv_customers={n_cust_dim}  dim_customers={n_cust_cust}')

=== INVOICE / CUSTOMER COUNTS ===
  fact_ii unique InvoiceIDs :   33,499
  dim_invoices rows          :   33,499
  Match                      : ✅ PASS

  Unique customers in dim_invoices :    8,237
  Rows in dim_customers            :    8,237
  All invoice customers in dim_cust: ✅ PASS
✅ [PASS] E1 invoice count fact_ii == dim_invoices
       fact_ii unique=33499  dim_invoices rows=33499
✅ [PASS] F1 all invoice customers in dim_customers
       inv_customers=8237  dim_customers=8237


## Section 10 — Save All Processed Tables

In [12]:
PROC = PROJECT_ROOT / 'data' / 'processed'
PROC.mkdir(parents=True, exist_ok=True)

dim_inv.to_csv(PROC  / 'dim_invoices.csv',       index=False)
fact_ii.to_csv(PROC  / 'fact_invoice_items.csv',  index=False)
dim_cust.to_csv(PROC / 'dim_customers.csv',       index=False)
dim_prod.to_csv(PROC / 'dim_products.csv',        index=False)

print('Saved:')
for f in ['dim_invoices.csv','fact_invoice_items.csv','dim_customers.csv','dim_products.csv']:
    p = PROC / f
    print(f'  {f:35s}  {p.stat().st_size/1024:.1f} KB')

Saved:
  dim_invoices.csv                     1129.0 KB
  fact_invoice_items.csv               28863.3 KB
  dim_customers.csv                    120.0 KB
  dim_products.csv                     216.2 KB


## Section 11 — Final Validation Summary

In [13]:
import json
from pathlib import Path

all_pass = all(c['passed'] for c in checks)

print('\n' + '='*70)
print('FINAL VALIDATION SUMMARY')
print('='*70)

print(f'\nDIM_INVOICES:')
print(f'  rows                = {len(dim_inv):,}')
print(f'  unique InvoiceID    = {dim_inv["InvoiceID"].nunique():,}')
print(f'  unique CustomerID   = {dim_inv["CustomerID"].nunique():,}')

print(f'\nFACT_INVOICE_ITEMS:')
print(f'  rows                = {len(fact_ii):,}')
print(f'  unique InvoiceID    = {fact_ii["InvoiceID"].nunique():,}')
print(f'  unique product_id   = {fact_ii["product_id"].nunique():,}')

print(f'\nDIM_CUSTOMERS:')
print(f'  rows                = {len(dim_cust):,}')

print(f'\nDIM_PRODUCTS:')
print(f'  rows                = {len(dim_prod):,}')

print(f'\nTOTAL REVENUE:')
print(f'  invoice_items direct    = £{rev_direct:>14,.2f}')
print(f'  dim_invoices aggregated = £{rev_dim_inv:>14,.2f}')
print(f'  difference              = £{rev_dim_inv - rev_direct:>14,.4f}')

print(f'\nALL VALIDATION CHECKS:')
for c in checks:
    icon = '✅ PASS' if c['passed'] else '❌ FAIL'
    print(f'  {icon}  {c["check"]}')

print(f'\nOVERALL: {"✅ ALL CHECKS PASSED" if all_pass else "❌ SOME CHECKS FAILED"}')

# Save validation report as JSON
report_data = {
    'dim_invoices':       {'rows': len(dim_inv), 'unique_InvoiceID': dim_inv['InvoiceID'].nunique(), 'unique_CustomerID': dim_inv['CustomerID'].nunique()},
    'fact_invoice_items': {'rows': len(fact_ii), 'unique_InvoiceID': fact_ii['InvoiceID'].nunique(), 'unique_product_id': fact_ii['product_id'].nunique()},
    'dim_customers':      {'rows': len(dim_cust)},
    'dim_products':       {'rows': len(dim_prod)},
    'revenue': {
        'invoice_items_direct': round(float(rev_direct), 2),
        'dim_invoices_agg':     round(float(rev_dim_inv), 2),
        'difference':           round(float(rev_dim_inv - rev_direct), 4),
    },
    'checks': checks,
    'all_passed': all_pass,
}
rpath = PROJECT_ROOT / 'reports' / 'data_model_validation_report_raw.json'
rpath.parent.mkdir(exist_ok=True)
(rpath).write_text(json.dumps(report_data, indent=2, default=str))
print(f'\nValidation data saved to: {rpath}')


FINAL VALIDATION SUMMARY

DIM_INVOICES:
  rows                = 33,499
  unique InvoiceID    = 33,499
  unique CustomerID   = 8,237

FACT_INVOICE_ITEMS:
  rows                = 431,263
  unique InvoiceID    = 33,499
  unique product_id   = 4,033

DIM_CUSTOMERS:
  rows                = 8,237

DIM_PRODUCTS:
  rows                = 4,033

TOTAL REVENUE:
  invoice_items direct    = £  9,307,314.40
  dim_invoices aggregated = £  9,307,314.40
  difference              = £        0.0000

ALL VALIDATION CHECKS:
  ✅ PASS  A1 dim_invoices InvoiceID unique
  ✅ PASS  A2 dim_invoices CustomerID not null
  ✅ PASS  A3 dim_invoices date not null
  ✅ PASS  A4 dim_invoices 1 customer per invoice
  ✅ PASS  A5 dim_invoices 1 date per invoice
  ✅ PASS  B1 fact_ii no exact duplicates
  ✅ PASS  B2 fact_ii InvoiceID not null
  ✅ PASS  B3 fact_ii product_id not null
  ✅ PASS  B4 fact_ii quantity >= 0
  ✅ PASS  B5 fact_ii price >= 0
  ✅ PASS  B6 fact_ii line_total >= 0
  ✅ PASS  C1 all dim_invoices IDs exist i